# Лабораторная работа №6. Квантовые сумматоры. Алгоритм Гровера

**Вариант 14.** Задание 2: `F(A,B,C,D,E) = ¬(A∧E) ∨ (B∧C∧D)`. Задание 7: `N = 2^6`, `|w⟩ = |010101⟩`.

Каждая задача печатает свой отчёт в консоль. Запускайте сверху вниз.

In [ ]:
import numpy as np, itertools, matplotlib.pyplot as plt
from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

backend = AerSimulator()
SEP = "=" * 78
def header(t): print("\n" + SEP + "\n" + t.center(78) + "\n" + SEP)
def sub(t): print("\n" + "-"*78 + "\n" + t + "\n" + "-"*78)
def conclusion(t): print("\n>>> ВЫВОД: " + t + "\n")

## ЗАДАНИЕ 1.1. Сумматор для A=1, B=0, X=1

In [ ]:
header("1.1. Полный сумматор для A=1, B=0, X=1")

def full_adder_5q():
    qc = QuantumCircuit(5)  # A,B,X,S,C
    qc.cx(0,3); qc.cx(1,3); qc.cx(2,3)   # S = A⊕B⊕X
    qc.ccx(0,1,4)                         # C = A·B
    qc.cx(0,1); qc.ccx(1,2,4); qc.cx(0,1) # C ^= (A⊕B)·X
    return qc

qc = full_adder_5q(); qc.x(0); qc.x(2)
sv = Statevector.from_instruction(qc)
idx = int(np.argmax(np.abs(sv.data)**2))
S_val = (idx >> 3) & 1; C_val = (idx >> 4) & 1
print("Базис: |q4 q3 q2 q1 q0⟩ = |" + format(idx, "05b") + "⟩")
print("Получено: S=" + str(S_val) + ", C=" + str(C_val))

creg = ClassicalRegister(2)
qm = qc.copy(); qm.add_register(creg); qm.measure(3,0); qm.measure(4,1)
counts = backend.run(transpile(qm, backend), shots=1024).result().get_counts()
print("Распределение (1024 shots):", counts)

conclusion("Для A=1,B=0,X=1 таблица требует S=0, C=1. Вектор состояния даёт единственную ненулевую амплитуду на базисе |10101⟩ (C=1,S=0,X=1,B=0,A=1). Измерения дают 100% исход '10' (старший бит — C, младший — S), что полностью согласуется с таблицей истинности.")

## ЗАДАНИЕ 1.2. Проверка 5-кубитового сумматора на всех 8 наборах

In [ ]:
header("1.2. Проверка 5-кубитового сумматора на 8 наборах")
print("{:<2}{:<2}{:<2} | {:<5}{:<5} | {:<5}{:<5} | статус".format("A","B","X","S_ож","C_ож","S_п","C_п"))
print("-" * 50)
all_ok = True
for a,b,x in itertools.product([0,1], repeat=3):
    qc = full_adder_5q()
    if a: qc.x(0)
    if b: qc.x(1)
    if x: qc.x(2)
    sv = Statevector.from_instruction(qc)
    idx = int(np.argmax(np.abs(sv.data)**2))
    s, c = (idx>>3)&1, (idx>>4)&1
    es, ec = (a+b+x)%2, (a+b+x)//2
    ok = (s==es) and (c==ec); all_ok &= ok
    assert ok, "Mismatch " + str((a,b,x))
    print("{:<2}{:<2}{:<2} | {:<5}{:<5} | {:<5}{:<5} | {}".format(a,b,x,es,ec,s,c,"OK" if ok else "FAIL"))
print("Все assert пройдены:", all_ok)

conclusion("Реализация из X, CX, CCX без ancilla и промежуточных измерений точно воспроизводит таблицу истинности полного сумматора на всех 8 входах.")

## ЗАДАНИЕ 1.3. Сумматор из двух полусумматоров и OR

In [ ]:
header("1.3. Сумматор из двух HA + OR (7 кубитов)")

def full_adder_2ha():
    qc = QuantumCircuit(7)  # A,B,X,S,C1,C2,Cout
    qc.cx(0,3); qc.ccx(0,1,4)               # HA1
    qc.ccx(3,2,5); qc.cx(2,3)               # HA2
    qc.ccx(4,5,6); qc.cx(4,5); qc.cx(5,6)   # OR
    return qc

print("{:<2}{:<2}{:<2} | {:<5}{:<5} | {:<5}{:<5} | статус".format("A","B","X","S_ож","C_ож","S_п","C_п"))
print("-" * 50)
all_ok = True
for a,b,x in itertools.product([0,1], repeat=3):
    qc = full_adder_2ha()
    if a: qc.x(0)
    if b: qc.x(1)
    if x: qc.x(2)
    sv = Statevector.from_instruction(qc)
    idx = int(np.argmax(np.abs(sv.data)**2))
    s, c = (idx>>3)&1, (idx>>6)&1
    es, ec = (a+b+x)%2, (a+b+x)//2
    ok = (s==es) and (c==ec); all_ok &= ok
    assert ok, "Mismatch " + str((a,b,x))
    print("{:<2}{:<2}{:<2} | {:<5}{:<5} | {:<5}{:<5} | {}".format(a,b,x,es,ec,s,c,"OK" if ok else "FAIL"))
print("Все assert пройдены:", all_ok)

conclusion("Сумматор из двух HA и OR корректен, но требует 7 кубитов (против 5) и больше вентилей, поэтому менее эффективен.")

## ЗАДАНИЕ 1.4. Сравнение двух реализаций

In [ ]:
header("1.4. Сравнение двух реализаций")

def stats(qc):
    o = qc.count_ops()
    return qc.num_qubits, o.get("cx",0), o.get("ccx",0), qc.depth()

print("{:<28}{:<8}{:<5}{:<5}{:<8}".format("Реализация","Кубиты","CX","CCX","Глубина"))
print("-" * 60)
for name, qc in [("5-кубитовый", full_adder_5q()), ("2 HA + OR (7)", full_adder_2ha())]:
    q,cx,ccx,d = stats(qc)
    print("{:<28}{:<8}{:<5}{:<5}{:<8}".format(name, q, cx, ccx, d))

conclusion("5-кубитовая реализация экономичнее по всем параметрам: 5 кубитов (против 7), 4 CX (против 6), 2 CCX (против 3), глубина 4 (против 6).")

## ЗАДАНИЕ 1.5. Ripple-carry adder

In [ ]:
header("1.5. Ripple-carry adder на n бит")

def build_ripple(n):
    qc = QuantumCircuit(4*n)
    A=lambda i:i; B=lambda i:n+i; S=lambda i:2*n+i; C=lambda i:3*n+(i-1)
    for i in range(n):
        qc.cx(A(i),S(i)); qc.cx(B(i),S(i))
        if i>0: qc.cx(C(i),S(i))
        qc.ccx(A(i),B(i),C(i+1))
        if i>0:
            qc.cx(A(i),B(i)); qc.ccx(B(i),C(i),C(i+1)); qc.cx(A(i),B(i))
    return qc

def test_ripple(n):
    base = build_ripple(n); ok_count = 0
    for a in range(2**n):
        for b in range(2**n):
            qc = base.copy()
            for i in range(n):
                if (a>>i)&1: qc.x(i)
                if (b>>i)&1: qc.x(n+i)
            sv = Statevector.from_instruction(qc)
            idx = int(np.argmax(np.abs(sv.data)**2))
            bits = format(idx, "0{}b".format(4*n))
            s = sum(int(bits[4*n-1-(2*n+i)])<<i for i in range(n))
            c = int(bits[0])<<n
            assert (c|s) == a+b, "FAIL n={} a={} b={}".format(n,a,b)
            ok_count += 1
    return ok_count

print("{:<3}{:<8}{:<6}{:<6}{:<10}{:<15}".format("n","Кубиты","CX","CCX","Глубина","Проверено пар"))
print("-" * 60)
for n in [2, 3, 4]:
    ok = test_ripple(n)
    q,cx,ccx,d = stats(build_ripple(n))
    print("{:<3}{:<8}{:<6}{:<6}{:<10}{:<15}".format(n, q, cx, ccx, d, ok))

conclusion("Для 4-битного сумматора все 256 пар (a,b) проверены программно через assert. Глубина растёт линейно (≈4n) — это классическая оценка для ripple-carry adder.")

### Ответы на вопросы к заданию 1

1. **Почему C нельзя вычислить одним CCX?** `C = (A∧B) ∨ (X∧(A⊕B))` — дизъюнкция двух конъюнкций. Один CCX даёт только одну; для второй нужен CX, для OR — ещё вентили.
2. **Связь CX с запретом клонирования.** CX копирует базисные |0⟩/|1⟩, но для суперпозиции даёт запутанное состояние `α|00⟩+β|11⟩` — не копию.
3. **Как уменьшить глубину 4-битного сумматора?** Carry-lookahead или QFT-сумматор — глубина падает, но растёт число ancilla и вентилей.

## ЗАДАНИЕ 2 (вариант 14). F = ¬(A∧E) ∨ (B∧C∧D)

In [ ]:
header("ЗАДАНИЕ 2. F(A,B,C,D,E) = ¬(A∧E) ∨ (B∧C∧D)")

def classical_F(A,B,C,D,E):
    return (1 - (A & E)) | (B & C & D)

def F_base():
    qc = QuantumCircuit(11)  # q0..q4 входы, q5 выход, q6..q10 ancilla
    qc.ccx(0,4,6)                     # q6 = A·E
    qc.cx(6,7); qc.x(7)               # q7 = ¬(A·E)
    qc.ccx(1,2,8)                     # q8 = B·C
    qc.ccx(8,3,9)                     # q9 = B·C·D
    qc.cx(7,5); qc.cx(9,5); qc.ccx(7,9,10); qc.cx(10,5)
    qc.ccx(7,9,10); qc.ccx(8,3,9); qc.ccx(1,2,8)     # uncompute
    qc.x(7); qc.cx(6,7); qc.ccx(0,4,6)
    return qc

def F_opt():
    qc = QuantumCircuit(10)  # на 1 ancilla меньше
    qc.ccx(0,4,6); qc.ccx(1,2,7); qc.ccx(7,3,8); qc.ccx(6,8,9)
    qc.cx(6,5); qc.cx(9,5); qc.x(5)   # q5 = 1 ⊕ AE ⊕ AE·BCD
    qc.ccx(6,8,9); qc.ccx(7,3,8); qc.ccx(1,2,7); qc.ccx(0,4,6)
    return qc

def test_F(builder):
    for bits in itertools.product([0,1], repeat=5):
        qc = builder()
        for i, b in enumerate(bits):
            if b: qc.x(i)
        sv = Statevector.from_instruction(qc)
        idx = int(np.argmax(np.abs(sv.data)**2))
        assert idx < 64, "Ancilla not clean: idx=" + str(idx)
        measured = (idx >> 5) & 1
        expected = classical_F(*bits)
        assert measured == expected, "Mismatch " + str(bits)
    return True

for name, bld in [("БАЗОВАЯ (11 кубитов)", F_base), ("ОПТИМИЗИРОВАННАЯ (10 кубитов)", F_opt)]:
    sub(name)
    rows = []
    for bits in itertools.product([0,1], repeat=5):
        qc = bld()
        for i, b in enumerate(bits):
            if b: qc.x(i)
        sv = Statevector.from_instruction(qc)
        idx = int(np.argmax(np.abs(sv.data)**2))
        m = (idx >> 5) & 1
        rows.append((bits, classical_F(*bits), m))
    print("Все 32 набора: статус OK (проверено assert), ancillas в |0⟩.")
    print("Пример: A=1 B=1 C=1 D=1 E=0 -> F_кл=1, F_кв=" + str(rows[-1][2]))

sub("Сравнение схем")
print("{:<24}{:<8}{:<5}{:<5}{:<4}{:<8}".format("Версия","Кубиты","CX","CCX","X","Глубина"))
for name, bld in [("Базовая", F_base), ("Оптимизированная", F_opt)]:
    qc = bld(); o = qc.count_ops()
    print("{:<24}{:<8}{:<5}{:<5}{:<4}{:<8}".format(name, qc.num_qubits, o.get("cx",0), o.get("ccx",0), o.get("x",0), qc.depth()))

conclusion("Обе версии проходят проверку на всех 32 входах, ancillas возвращаются в |0⟩. Оптимизированная использует 10 кубитов вместо 11, 4 CCX вместо 5, 2 CX вместо 4, 1 X вместо 2 — за счёт тождества ¬(AE) ∨ BCD = 1 ⊕ AE ⊕ (AE·BCD). Минимумы по кубитам и по глубине могут не совпадать: экономия ancilla требует последовательных операций.")

## ЗАДАНИЕ 3. Шум

In [ ]:
header("ЗАДАНИЕ 3. Шум на CX")

def noise_sim(builder, p2_list, shots=200):
    fracs = []
    for p2 in p2_list:
        nm = NoiseModel()
        nm.add_all_qubit_quantum_error(depolarizing_error(0.001,1), ["x","h","sx","rz"])
        if p2 > 0:
            nm.add_all_qubit_quantum_error(depolarizing_error(p2,2), ["cx"])
        correct = 0
        for bits in itertools.product([0,1], repeat=5):
            qc = builder()
            for i, b in enumerate(bits):
                if b: qc.x(i)
            creg = ClassicalRegister(1)
            qm = qc.copy(); qm.add_register(creg); qm.measure(5,0)
            counts = backend.run(transpile(qm, backend), shots=shots, noise_model=nm).result().get_counts()
            if int(max(counts, key=counts.get), 2) == classical_F(*bits):
                correct += 1
        fracs.append(correct / 32)
    return fracs

p2_list = [0, 0.001, 0.005, 0.01, 0.02]
f_base = noise_sim(F_base, p2_list)
f_opt  = noise_sim(F_opt,  p2_list)
print("{:<8}{:<12}{:<20}".format("p2","Базовая","Оптимизированная"))
print("-" * 40)
for p2, fb, fo in zip(p2_list, f_base, f_opt):
    print("{:<8}{:<12.3f}{:<20.3f}".format(p2, fb, fo))

plt.figure(figsize=(7,4))
plt.plot(p2_list, f_base, "o-", label="базовая")
plt.plot(p2_list, f_opt,  "s-", label="оптимизированная")
plt.xlabel("p2"); plt.ylabel("доля верных"); plt.title("Задание 3"); plt.legend(); plt.grid(True); plt.show()

conclusion("Доля правильных ответов падает с ростом p2 тем быстрее, чем больше CX в схеме. Оптимизированная версия устойчивее: меньше CX/CCX — меньше ошибок.")

## ЗАДАНИЕ 4. Оракул и диффузор

In [ ]:
header("ЗАДАНИЕ 4. Оракул, диффузор, Гровер")

def build_oracle(n, marked):
    qc = QuantumCircuit(n)
    for i, b in enumerate(marked[::-1]):
        if b == "0": qc.x(i)
    qc.h(n-1)
    if n>=2: qc.mcx(list(range(n-1)), n-1)
    else: qc.z(0)
    qc.h(n-1)
    for i, b in enumerate(marked[::-1]):
        if b == "0": qc.x(i)
    return qc

def build_diffuser(n):
    qc = QuantumCircuit(n)
    qc.h(range(n)); qc.x(range(n)); qc.h(n-1)
    if n>=2: qc.mcx(list(range(n-1)), n-1)
    else: qc.z(0)
    qc.h(n-1); qc.x(range(n)); qc.h(range(n))
    return qc

def build_grover(n, marked, k):
    qc = QuantumCircuit(n); qc.h(range(n))
    for _ in range(k):
        qc.compose(build_oracle(n, marked), inplace=True)
        qc.compose(build_diffuser(n), inplace=True)
    return qc

def grover_prob(n, marked, k):
    return abs(Statevector.from_instruction(build_grover(n, marked, k)).data[int(marked,2)])**2

sub("Проверка оракула")
for n in [2,3,4]:
    marked = format(2, "0{}b".format(n))
    qc = build_oracle(n, marked); ok = True
    for idx in range(2**n):
        qt = QuantumCircuit(n)
        for i in range(n):
            if (idx>>i)&1: qt.x(i)
        qt.compose(qc, inplace=True)
        amp = Statevector.from_instruction(qt).data[idx]
        target = -1 if idx==int(marked,2) else 1
        ok &= np.isclose(amp, target)
    print("n={}: {}".format(n, "OK" if ok else "FAIL"))

sub("Сверка с sin² для n=3")
n=3; marked="101"; theta = np.arcsin(1/np.sqrt(2**n))
for k in [1,2]:
    th = np.sin((2*k+1)*theta)**2
    qv = grover_prob(n, marked, k)
    print("k={}: теория={:.6f}, Qiskit={:.6f}, Δ={:.2e}".format(k, th, qv, abs(th-qv)))

conclusion("Оракул инвертирует фазу только |marked⟩ — проверено для n=2,3,4. Диффузор реализует 2|s⟩⟨s|−I с точностью до знака. Вероятность совпадает с sin²((2k+1)θ).")

## ЗАДАНИЕ 5. P(k) для N=2^7

In [ ]:
header("ЗАДАНИЕ 5. N=2^7, |w⟩=|1010011⟩")

n=7; marked="1010011"; N=2**n; theta=np.arcsin(1/np.sqrt(N)); Ks=list(range(16))
p_exact = [grover_prob(n, marked, k) for k in Ks]
p_exp = []
for k in Ks:
    qc = build_grover(n, marked, k)
    creg = ClassicalRegister(n)
    qm = qc.copy(); qm.add_register(creg); qm.measure(range(n), range(n))
    c = backend.run(transpile(qm, backend), shots=1024).result().get_counts()
    p_exp.append(c.get(marked, 0)/1024)

print("{:<4}{:<10}{:<10}{:<10}".format("k","P_точн","P_эксп","sin²"))
for k in Ks:
    print("{:<4}{:<10.4f}{:<10.4f}{:<10.4f}".format(k, p_exact[k], p_exp[k], np.sin((2*k+1)*theta)**2))

k_th = int(np.floor(np.pi/4*np.sqrt(N)))
print("\nk_теор =", k_th, "| k_точн_макс =", int(np.argmax(p_exact)), "| k_эксп_макс =", int(np.argmax(p_exp)))
print("max P_точн = {:.4f}".format(max(p_exact)))

plt.figure(figsize=(8,4))
plt.plot(Ks, p_exact, "o-", label="точная")
plt.plot(Ks, p_exp, "s--", label="эксп 1024")
plt.plot(Ks, [np.sin((2*k+1)*theta)**2 for k in Ks], ":", label="sin²")
plt.xlabel("k"); plt.ylabel("P"); plt.legend(); plt.grid(True); plt.title("Задание 5"); plt.show()

conclusion("Теоретический оптимум floor(π/4·√N) для N=128 совпадает с точным максимумом (k=8, P≈0.996). Экспериментальный максимум при 1024 shots может сдвинуться на соседнее k из-за флуктуаций ~1%.")

## ЗАДАНИЕ 6. M решений

In [ ]:
header("ЗАДАНИЕ 6. Зависимость от M")

def build_oracle_multi(n, ml):
    qc = QuantumCircuit(n)
    for w in ml: qc.compose(build_oracle(n, w), inplace=True)
    return qc

N=2**6; np.random.seed(42)
print("{:<4}{:<8}{:<9}{:<8}".format("M","k_теор","k_1макс","P_max"))
print("-"*35)
for M in [1,2,3,4,8]:
    ml=[]
    while len(ml)<M:
        c = format(np.random.randint(0,N),"06b")
        if c not in ml: ml.append(c)
    p_k=[]
    for k in range(16):
        qc = QuantumCircuit(6); qc.h(range(6))
        for _ in range(k):
            qc.compose(build_oracle_multi(6,ml), inplace=True)
            qc.compose(build_diffuser(6), inplace=True)
        sv = Statevector.from_instruction(qc)
        p_k.append(sum(abs(sv.data[int(w,2)])**2 for w in ml))
    k1 = int(np.argmax(p_k)); kt = int(np.floor(np.pi/4*np.sqrt(N/M)))
    print("{:<4}{:<8}{:<9}{:<8.4f}".format(M, kt, k1, max(p_k)))

conclusion("Первый локальный максимум P(k) хорошо согласуется с floor(π/4·√(N/M)). При больших M (N/2, 3N/4) оптимум 0–1 итерация — выигрыша нет. Гровер полезен только при M ≪ N.")

## ЗАДАНИЕ 7 (вариант 14). N=2^6, |w⟩=|010101⟩

In [ ]:
header("ЗАДАНИЕ 7. Вариант 14: N=2^6, |w⟩=|010101⟩")

n=6; marked="010101"; N=2**n; theta=np.arcsin(1/np.sqrt(N)); Ks=list(range(16))
p_exact = [grover_prob(n, marked, k) for k in Ks]
p_exp = []
for k in Ks:
    qc = build_grover(n, marked, k)
    creg = ClassicalRegister(n)
    qm = qc.copy(); qm.add_register(creg); qm.measure(range(n), range(n))
    c = backend.run(transpile(qm, backend), shots=1024).result().get_counts()
    p_exp.append(c.get(marked, 0)/1024)

print("{:<4}{:<10}{:<10}{:<10}".format("k","P_точн","P_эксп","sin²"))
for k in Ks:
    print("{:<4}{:<10.4f}{:<10.4f}{:<10.4f}".format(k, p_exact[k], p_exp[k], np.sin((2*k+1)*theta)**2))

k_th = int(np.floor(np.pi/4*np.sqrt(N)))
print("\nk_теор =", k_th, "| k_точн =", int(np.argmax(p_exact)), "| k_эксп =", int(np.argmax(p_exp)), "| P_max = {:.4f}".format(max(p_exact)))

plt.figure(figsize=(8,4))
plt.plot(Ks, p_exact, "o-", label="точная")
plt.plot(Ks, p_exp, "s--", label="эксп 1024")
plt.plot(Ks, [np.sin((2*k+1)*theta)**2 for k in Ks], ":", label="sin²")
plt.xlabel("k"); plt.ylabel("P"); plt.legend(); plt.grid(True); plt.title("Задание 7, вариант 14"); plt.show()

conclusion("Для N=64, |010101⟩: k_теор=6, P_max≈0.99. Оптимум зависит только от N, а не от конкретного |w⟩.")

## ЗАДАНИЕ 7.4. Шум на CX

In [ ]:
header("ЗАДАНИЕ 7.4. Шум на CX (p2=0.005 и 0.02)")

def noisy_grover(n, marked, p2, shots=1024):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(0.001,1), ["x","h","sx","rz"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p2,2), ["cx"])
    p=[]
    for k in range(16):
        qc = build_grover(n, marked, k)
        creg = ClassicalRegister(n)
        qm = qc.copy(); qm.add_register(creg); qm.measure(range(n), range(n))
        c = backend.run(transpile(qm, backend), shots=shots, noise_model=nm).result().get_counts()
        p.append(c.get(marked, 0)/shots)
    return p

plt.figure(figsize=(8,4))
for p2 in [0.005, 0.02]:
    pn = noisy_grover(6, "010101", p2)
    plt.plot(range(16), pn, "o--", label="p2={}".format(p2))
    print("p2={}: k_макс={}, P_max={:.4f}".format(p2, int(np.argmax(pn)), max(pn)))
plt.plot(Ks, p_exact, "k-", label="без шума")
plt.xlabel("k"); plt.ylabel("P"); plt.legend(); plt.grid(True); plt.title("Задание 7.4"); plt.show()

conclusion("Шум снижает пиковое значение P и может сдвигать оптимум к меньшим k. Глубина схемы напрямую определяет накопление ошибок.")